In [11]:
#!/usr/bin/env python3
"""
Second-stage post-processing: reads existing vol_sm GeoTIFFs (from an
earlier run of the tif-based pipeline) and derives IMD and remaining
upper-zone capacity, without re-touching any NetCDF/resampling logic for
the raw soil moisture data.

Writes a single combined netCDF per event -- same {vol_sm, imd,
remaining_capacity} format as soil_pipeline.py's production output -- so
downstream code can treat outputs from both scripts identically regardless
of which one produced them.

Expects the existing directory structure:
    <VOL_SM_ROOT>/<ensemble>/tif/<HA_NUM>/soil_<HA_NUM>_<event>_<y>_<m>_<d>_<ensemble>_vol_30m.tif

Writes:
    <OUTPUT_NC_DIR>/<ensemble>/nc/<HA_NUM>/..._vol_30m.nc
"""
import argparse
import glob
import logging
import os
from datetime import datetime
from tqdm import tqdm

import numpy as np
import rasterio
import rioxarray
import xarray as xr
from shapely.geometry import box

from soil_pipeline import (CRS_TARGET,POROSITY_CSV,SOIL_TEXTURE_DIR,STATIC_LU_PATH,array_to_dataarray,
                           load_porosity_grids,resample_to_30m_array,)

VOL_SM_ROOT = ("/scratch/hydro5/users/la17355/FUTURE-FLOOD/UKCP_soil_moisture/volumetric_30m/soil_depth_missing_tests/")

# NOTE: matches the production pipeline's OUTPUT_VOL_DIR location
# (kv25483's scratch space) -- double check this is deliberate.
OUTPUT_NC_DIR = "/scratch/hydro4/users/kv25483/FutureFlood/Data/SoilDynamicVariables/combined_from_tif/"

LOG_DIR = "/scratch/hydro4/users/kv25483/FutureFlood"


def build_dem_info_from_tif(tif_path):
    """Grid info read directly from the vol_sm tif itself, so IMD/capacity
    are guaranteed to land on the exact same grid vol_sm was written on --
    no risk of drift if resampling parameters ever change upstream."""
    with rasterio.open(tif_path) as src:
        return {"transform": src.transform,"crs": src.crs,"width": src.width,"height": src.height,"bounds": src.bounds,}


def load_static_grids_for_ha(ha_num, dem_info):
    """Porosity + Lu, both clipped and resampled onto this exact grid.

    Porosity is resampled here rather than used at its native resolution,
    because the native soil-texture raster and the DEM-derived vol_sm grid
    are two independently-built products with no guaranteed pixel-for-pixel
    alignment -- confirmed earlier by their bounding boxes not overlapping
    at all for a mismatched HA_NUM. Always resample onto dem_info, never
    assume native alignment.
    """
    porosity_grids = load_porosity_grids(POROSITY_CSV, SOIL_TEXTURE_DIR, {ha_num})
    if ha_num not in porosity_grids:
        return None, None

    entry = porosity_grids[ha_num]
    porosity_da = ( xr.DataArray(entry["grid"], dims=("y", "x"))
        .rio.write_crs(entry["crs"])
        .rio.write_transform(entry["transform"]))
    porosity_data, _ = resample_to_30m_array(porosity_da, dem_info)

    lu_national = rioxarray.open_rasterio(STATIC_LU_PATH, masked=True).squeeze()
    if not lu_national.rio.crs:
        lu_national = lu_national.rio.write_crs(CRS_TARGET)
    bounds_geom = box(*dem_info["bounds"])
    lu_clipped = lu_national.rio.clip_box(*bounds_geom.bounds)
    lu_30m, _ = resample_to_30m_array(lu_clipped, dem_info)

    return porosity_data, lu_30m


def compute_imd_and_capacity(vol_sm_path, porosity_data, lu_data):
    """Compute IMD and remaining capacity arrays from an existing vol_sm
    tif. Returns arrays plus the grid metadata needed to write them out --
    no file writing here, so this is also usable directly for quick
    interactive inspection (e.g. checking value ranges before writing
    anything to disk)."""
    with rasterio.open(vol_sm_path) as src:
        vol_sm = src.read(1).astype(np.float32)
        transform = src.transform
        crs = src.crs
        nodata = src.nodata if src.nodata is not None else -9999

    valid = vol_sm != nodata
    with np.errstate(divide="ignore", invalid="ignore"):
        imd = np.where(valid, porosity_data * (1.0 - vol_sm), nodata)
        remaining_capacity = np.where(valid, imd * lu_data, nodata)

    imd = np.round(imd, 4).astype(np.float32)
    remaining_capacity = np.round(remaining_capacity, 4).astype(np.float32)
    imd[~valid] = nodata
    remaining_capacity[~valid] = nodata

    return vol_sm, imd, remaining_capacity, transform, crs, nodata


def write_combined_nc(vol_sm, imd, remaining_capacity, transform, crs, nodata, nc_out):
    """Write vol_sm, imd, and remaining_capacity as three variables in a
    single combined netCDF, matching soil_pipeline.py's output format."""
    ds = xr.Dataset({"vol_sm": array_to_dataarray(vol_sm, transform, crs, "vol_sm", nodata),
            "imd": array_to_dataarray(imd, transform, crs, "imd", nodata),
            "remaining_capacity": array_to_dataarray(remaining_capacity, transform, crs, "remaining_capacity", nodata), })
    ds["vol_sm"].attrs["units"] = "1"
    ds["imd"].attrs["units"] = "1"
    ds["remaining_capacity"].attrs["units"] = "m"

    os.makedirs(os.path.dirname(nc_out), exist_ok=True)
    ds.to_netcdf(nc_out)


def derive_and_write(vol_sm_path, porosity_data, lu_data, nc_out):
    """Combined compute + write, for use inside the batch loop below."""
    vol_sm, imd, remaining_capacity, transform, crs, nodata = compute_imd_and_capacity(
        vol_sm_path, porosity_data, lu_data)
    write_combined_nc(vol_sm, imd, remaining_capacity, transform, crs, nodata, nc_out)


def run_second_stage(target_has=None):
    """Scan VOL_SM_ROOT and derive IMD/capacity for every matching tif.
    If target_has is given (a set of HA_NUM strings), only process those.
    """

    static_cache = {}  # ha_num -> (porosity_data, lu_data)

    pattern = os.path.join(VOL_SM_ROOT, "*", "tif", "*", "*_vol_30m.tif")
    for vol_sm_path in tqdm(glob.glob(pattern)):
        parts = vol_sm_path.split(os.sep)
        ha_num = parts[-2]
        ensemble = parts[-4]

        fname = os.path.basename(vol_sm_path)
        event_base = fname.replace("_vol_30m.tif", "")
        nc_out = os.path.join(OUTPUT_NC_DIR, ensemble, "nc", ha_num, f"{event_base}_vol_30m.nc")

        dem_info = build_dem_info_from_tif(vol_sm_path)

        if ha_num not in static_cache:
            porosity_data, lu_data = load_static_grids_for_ha(ha_num, dem_info)
            if porosity_data is None:
                logging.error("[Porosity Missing] %s", ha_num)
                continue
            static_cache[ha_num] = (porosity_data, lu_data)

        porosity_data, lu_data = static_cache[ha_num]
        derive_and_write(vol_sm_path, porosity_data, lu_data, nc_out)
        
run_second_stage("40")

  4%|█▏                             | 224/6037 [10:11<4:24:23,  2.73s/it]


KeyboardInterrupt: 

In [4]:
#!/usr/bin/env python3
"""
Second-stage post-processing: reads existing vol_sm GeoTIFFs (already
produced by the saturation pipeline) and derives IMD and remaining
upper-zone capacity, without re-touching any NetCDF/resampling logic.

Expects the existing directory structure:
    <VOL_SM_ROOT>/<ensemble>/tif/<HA_NUM>/soil_<HA_NUM>_<event>_<y>_<m>_<d>_<ensemble>_vol_30m.tif

Writes:
    <OUTPUT_IMD_DIR>/<ensemble>/tif/<HA_NUM>/..._imd_30m.tif
    <OUTPUT_CAPACITY_DIR>/<ensemble>/tif/<HA_NUM>/..._capacity_30m.tif
"""
import glob
import logging
import os
import re
from datetime import datetime

import numpy as np
import rasterio
import rioxarray
import xarray as xr
from shapely.geometry import box

from soil_pipeline import (CRS_TARGET,POROSITY_CSV,SOIL_TEXTURE_DIR,STATIC_LU_PATH,load_porosity_grids,
                           resample_to_30m_array,)

VOL_SM_ROOT = ("/scratch/hydro5/users/la17355/FUTURE-FLOOD/UKCP_soil_moisture/"
    "volumetric_30m/soil_depth_missing_tests/")

# OUTPUT_IMD_DIR = '/scratch/hydro4/users/kv25483/FutureFlood/Data/imd_30m/soil_depth_missing_tests/'
# OUTPUT_CAPACITY_DIR = '/scratch/hydro4/users/kv25483/FutureFlood/Data/remaining_capacity_30m/soil_depth_missing_tests/'

def build_dem_info_from_tif(tif_path):
    """Grid info read directly from the vol_sm tif itself, so IMD/capacity
    are guaranteed to land on the exact same grid vol_sm was written on --
    no risk of drift if resampling parameters ever change upstream."""
    with rasterio.open(tif_path) as src:
        return { "transform": src.transform,"crs": src.crs,"width": src.width,
            "height": src.height, "bounds": src.bounds,}


# def load_static_grids_for_ha(ha_num, dem_info):
#     """Porosity + Lu, clipped onto this exact grid. Cache these per ha_num
#     at the call site rather than reloading per file."""
#     porosity_grids = load_porosity_grids(POROSITY_CSV, SOIL_TEXTURE_DIR, {ha_num})
#     if ha_num not in porosity_grids:
#         return None, None
#     porosity_data = porosity_grids[ha_num]["grid"]

#     lu_national = rioxarray.open_rasterio(STATIC_LU_PATH, masked=True).squeeze()
#     if not lu_national.rio.crs:
#         lu_national = lu_national.rio.write_crs(CRS_TARGET)
#     from shapely.geometry import box
#     bounds_geom = box(*dem_info["bounds"])
#     lu_clipped = lu_national.rio.clip_box(*bounds_geom.bounds)
#     lu_30m, _ = resample_to_30m_array(lu_clipped, dem_info)

#     return porosity_data, lu_30m

def load_static_grids_for_ha(ha_num, dem_info):
    """Porosity + Lu, both clipped and resampled onto this exact grid."""
    porosity_grids = load_porosity_grids(POROSITY_CSV, SOIL_TEXTURE_DIR, {ha_num})
    if ha_num not in porosity_grids:
        return None, None

    entry = porosity_grids[ha_num]
    porosity_da = xr.DataArray(
        entry["grid"], dims=("y", "x")
    ).rio.write_crs(entry["crs"]).rio.write_transform(entry["transform"])

    porosity_data, _ = resample_to_30m_array(porosity_da, dem_info)

    lu_national = rioxarray.open_rasterio(STATIC_LU_PATH, masked=True).squeeze()
    if not lu_national.rio.crs:
        lu_national = lu_national.rio.write_crs(CRS_TARGET)
    bounds_geom = box(*dem_info["bounds"])
    lu_clipped = lu_national.rio.clip_box(*bounds_geom.bounds)
    lu_30m, _ = resample_to_30m_array(lu_clipped, dem_info)

    return porosity_data, lu_30m


def derive_imd_and_capacity(vol_sm_path, porosity_data, lu_data, imd_out, capacity_out):
    with rasterio.open(vol_sm_path) as src:
        vol_sm = src.read(1).astype(np.float32)
        meta = src.meta.copy()
        nodata = src.nodata if src.nodata is not None else -9999

    valid = vol_sm != nodata
    with np.errstate(divide="ignore", invalid="ignore"):
        imd = np.where(valid, porosity_data * (1.0 - vol_sm), nodata)
        remaining_capacity = np.where(valid, imd * lu_data, nodata)

    imd = np.round(imd, 4).astype(np.float32)
    remaining_capacity = np.round(remaining_capacity, 4).astype(np.float32)
    imd[~valid] = nodata
    remaining_capacity[~valid] = nodata

    meta.update(dtype=rasterio.float32, nodata=nodata, compress="LZW", driver="GTiff")

    for array, out_path in [(imd, imd_out), (remaining_capacity, capacity_out)]:
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(array, 1)


def run_second_stage(target_has=None):
    """Scan VOL_SM_ROOT and derive IMD/capacity for every matching tif.
    If target_has is given (a set of HA_NUM strings), only process those.
    """
    log_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs(LOG_DIR, exist_ok=True)
    logging.basicConfig(filename=os.path.join(LOG_DIR, f"imd_capacity_stage2_{log_time}.txt"),
        level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s",)

    static_cache = {}  # ha_num -> (porosity_data, lu_data), keyed per grid shape

    pattern = os.path.join(VOL_SM_ROOT, "*", "tif", "*", "*_vol_30m.tif")
    for vol_sm_path in glob.glob(pattern):
        parts = vol_sm_path.split(os.sep)
        ha_num = parts[-2]
        ensemble = parts[-4]

        if target_has and ha_num not in target_has:
            continue

        fname = os.path.basename(vol_sm_path)
        event_base = fname.replace("_vol_30m.tif", "")

        imd_out = os.path.join(OUTPUT_IMD_DIR, ensemble, "tif", ha_num, f"{event_base}_imd_30m.tif")
        capacity_out = os.path.join(OUTPUT_CAPACITY_DIR, ensemble, "tif", ha_num, f"{event_base}_capacity_30m.tif")

        try:
            dem_info = build_dem_info_from_tif(vol_sm_path)

            if ha_num not in static_cache:
                porosity_data, lu_data = load_static_grids_for_ha(ha_num, dem_info)
                if porosity_data is None:
                    logging.error("[Porosity Missing] %s", ha_num)
                    continue
                static_cache[ha_num] = (porosity_data, lu_data)

            porosity_data, lu_data = static_cache[ha_num]
            derive_imd_and_capacity(vol_sm_path, porosity_data, lu_data, imd_out, capacity_out)

        except Exception as exc:
            logging.error("[Stage2 Error] %s: %s", vol_sm_path, exc)


from tqdm import tqdm

static_cache = {}  # ha_num -> (porosity_data, lu_data), keyed per grid shape

pattern = os.path.join(VOL_SM_ROOT, "*", "tif", "*", "*_vol_30m.tif")
for vol_sm_path in tqdm(glob.glob(pattern)[:2]):
    parts = vol_sm_path.split(os.sep)
    ha_num = parts[-2]
    ensemble = parts[-4]

#     if target_has and ha_num not in target_has:
#         continue
    ha_num = "40"

    fname = os.path.basename(vol_sm_path)
    event_base = fname.replace("_vol_30m.tif", "")

    #imd_out = os.path.join(OUTPUT_IMD_DIR, ensemble, "tif", ha_num, f"{event_base}_imd_30m.tif")
    #capacity_out = os.path.join(OUTPUT_CAPACITY_DIR, ensemble, "tif", ha_num, f"{event_base}_capacity_30m.tif")

    dem_info = build_dem_info_from_tif(vol_sm_path)

    if ha_num not in static_cache:
        porosity_data, lu_data = load_static_grids_for_ha(ha_num, dem_info)
        if porosity_data is None:
            print("porosity missing")
            logging.error("[Porosity Missing] %s", ha_num)
            continue
        static_cache[ha_num] = (porosity_data, lu_data)
    porosity_data, lu_data = static_cache[ha_num]
    derive_imd_and_capacity(vol_sm_path, porosity_data, lu_data, imd_out, capacity_out)



  0%|                                                                                               | 0/2 [00:04<?, ?it/s]


NameError: name 'imd_out' is not defined

In [5]:
from tqdm import tqdm

static_cache = {}  # ha_num -> (porosity_data, lu_data), keyed per grid shape

pattern = os.path.join(VOL_SM_ROOT, "*", "tif", "*", "*_vol_30m.tif")
for vol_sm_path in tqdm(glob.glob(pattern)[:1]):
    parts = vol_sm_path.split(os.sep)
    ha_num = parts[-2]
    ensemble = parts[-4]

#     if target_has and ha_num not in target_has:
#         continue
    ha_num = "40"

    fname = os.path.basename(vol_sm_path)
    event_base = fname.replace("_vol_30m.tif", "")

    #imd_out = os.path.join(OUTPUT_IMD_DIR, ensemble, "tif", ha_num, f"{event_base}_imd_30m.tif")
    #capacity_out = os.path.join(OUTPUT_CAPACITY_DIR, ensemble, "tif", ha_num, f"{event_base}_capacity_30m.tif")

    dem_info = build_dem_info_from_tif(vol_sm_path)

    if ha_num not in static_cache:
        porosity_data, lu_data = load_static_grids_for_ha(ha_num, dem_info)
        if porosity_data is None:
            print("porosity missing")
            logging.error("[Porosity Missing] %s", ha_num)
            continue
        static_cache[ha_num] = (porosity_data, lu_data)
    porosity_data, lu_data = static_cache[ha_num]
    derive_imd_and_capacity(vol_sm_path, porosity_data, lu_data, imd_out, capacity_out)



  0%|                                                                                                             | 0/1 [00:05<?, ?it/s]


NameError: name 'imd_out' is not defined

In [ ]:
import rasterio

for label, path in [
    ("vol_sm", vol_sm_path),
    ("soil_texture", glob.glob(os.path.join(SOIL_TEXTURE_DIR, f"*{ha_num}*.tif"))[0]),
]:
    with rasterio.open(path) as src:
        print(label, src.shape, src.bounds, src.crs)